<a href="https://colab.research.google.com/github/swiz9/Flight-Delay-Prediction-System/blob/ann-model/FNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Mount Google Drive to access files stored in your Drive from Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:

#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from google.colab import drive
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [6]:
# Mount the Google Drive folder to access files from Drive in Colab
drive.mount('/content/drive')
# Load the flight dataset from Google Drive and display the first five rows
path = "/content/drive/MyDrive/flights_sample_3m.zip"
df = pd.read_csv(path)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,0.0,186.0,176.0,153.0,1065.0,NaN,NaN,NaN,NaN,NaN
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,0.0,235.0,236.0,189.0,1399.0,NaN,NaN,NaN,NaN,NaN
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,0.0,118.0,112.0,87.0,680.0,NaN,NaN,NaN,NaN,NaN
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",...,0.0,260.0,285.0,249.0,1589.0,0.0,0.0,24.0,0.0,0.0
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",...,0.0,181.0,182.0,153.0,985.0,NaN,NaN,NaN,NaN,NaN


In [7]:
# Remove irrelevant or unnecessary columns from the dataset to simplify analysis
def drop_irrelevant_columns(data):
    columns_to_drop = ['CANCELLED', 'CANCELLATION_CODE', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON',
                       'TAXI_IN', 'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS',
                       'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT', 'DOT_CODE', 'AIRLINE_CODE',
                       'ORIGIN_CITY', 'DEST_CITY', 'AIRLINE_DOT', 'FL_NUMBER', 'DIVERTED']
    return data.drop(columns=[col for col in columns_to_drop if col in data.columns], axis=1)

# Extract useful time-based features (day, month, hour, etc.) from the flight date column
def extract_date_features(data):
    if 'FL_DATE' in data.columns:
        data['FL_DATE'] = pd.to_datetime(data['FL_DATE'], errors='coerce')
        data['DayOfWeek'] = data['FL_DATE'].dt.dayofweek
        data['Month'] = data['FL_DATE'].dt.month
        data['Day'] = data['FL_DATE'].dt.day
        data['Hour'] = data['CRS_DEP_TIME'].astype(str).str.zfill(4).str[:2].astype(int)
        data.drop(columns=['FL_DATE'], inplace=True)  # Drop original date column
    return data